# Lazy Expressions 🚀

> Abstract Syntax Tree (AST) for column-level operations in MXFrame.

In [ ]:
#| default_exp lazy_expr

In [ ]:
#| export
from typing import Any, List, Optional, Union
from dataclasses import dataclass

## The `Expr` Class 🌳

The `Expr` class represents a node in our expression tree. It allows users to build complex column operations without executing them immediately.

In [ ]:
#| export
class Expr:
    """Base class for all lazy expressions."""
    def __init__(self, op: str, *args, **kwargs):
        self.op = op
        self.args = args
        self.kwargs = kwargs
        self._alias: Optional[str] = None

    def alias(self, name: str) -> 'Expr':
        """Rename the output of this expression."""
        self._alias = name
        return self

    def __add__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("add", self, lit(other) if not isinstance(other, Expr) else other)

    def __sub__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("sub", self, lit(other) if not isinstance(other, Expr) else other)

    def __mul__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("mul", self, lit(other) if not isinstance(other, Expr) else other)

    def __truediv__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("div", self, lit(other) if not isinstance(other, Expr) else other)

    def __eq__(self, other: Union['Expr', Any]) -> 'Expr': # type: ignore
        return Expr("eq", self, lit(other) if not isinstance(other, Expr) else other)

    def __gt__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("gt", self, lit(other) if not isinstance(other, Expr) else other)

    def __lt__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("lt", self, lit(other) if not isinstance(other, Expr) else other)

    def __ge__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("ge", self, lit(other) if not isinstance(other, Expr) else other)

    def __le__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("le", self, lit(other) if not isinstance(other, Expr) else other)

    def __and__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("and", self, lit(other) if not isinstance(other, Expr) else other)

    def __or__(self, other: Union['Expr', Any]) -> 'Expr':
        return Expr("or", self, lit(other) if not isinstance(other, Expr) else other)

    def sum(self) -> 'Expr':
        return Expr("sum", self)

    def min(self) -> 'Expr':
        return Expr("min", self)

    def max(self) -> 'Expr':
        return Expr("max", self)

    def mean(self) -> 'Expr':
        return Expr("mean", self)

    def count(self) -> 'Expr':
        """Count the number of rows. After a filter, reflects filtered row count."""
        return Expr("count", self)

    def __repr__(self) -> str:
        args_str = ", ".join(repr(a) for a in self.args)
        alias_str = f" AS {self._alias}" if self._alias else ""
        return f"{self.op}({args_str}){alias_str}"


## Helper Functions 🛠️

Functions to easily create column references and literal values.

In [ ]:
#| export
def col(name: str) -> Expr:
    """Create a column reference expression."""
    return Expr("col", name)

def lit(value: Any) -> Expr:
    """Create a literal value expression."""
    return Expr("lit", value)

## Tests 🧪

Let's verify that our AST builds correctly.

In [ ]:
e1 = col("a") + lit(1)
assert e1.op == "add"
assert e1.args[0].op == "col"
assert e1.args[0].args[0] == "a"
assert e1.args[1].op == "lit"
assert e1.args[1].args[0] == 1
print("\u2705 Test 1 passed: addition expr")

e2 = (col("b") * 2).alias("b_doubled")
assert e2.op == "mul"
assert e2._alias == "b_doubled"
print("\u2705 Test 2 passed: multiply + alias")

e3 = col("c").sum()
assert e3.op == "sum"
assert e3.args[0].op == "col"
print("\u2705 Test 3 passed: sum reduction")

e4 = col("x").count()
assert e4.op == "count"
assert e4.args[0].op == "col"
assert e4.args[0].args[0] == "x"
print("\u2705 Test 4 passed: count reduction")

e5 = (col("a") > lit(1)) & (col("b") < lit(5))
assert e5.op == "and"
assert e5.args[0].op == "gt"
assert e5.args[1].op == "lt"
print("\u2705 Test 5 passed: boolean combinator (&)")

print("\nAll lazy_expr tests passed! \U0001f389")


All tests passed! 🎉
